In [36]:
setwd("/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb")
library(devtools)
library(data.table)
library(reshape2)

devtools::load_all("utils/modules/R/phasingtools/")

Loading required package: usethis

i Loading phasingtools

Loading required package: ggplot2

Loading required package: Hmisc

Loading required package: lattice

Loading required package: survival

Loading required package: Formula


Attaching package: 'Hmisc'


The following objects are masked from 'package:base':

    format.pval, units


Loading required package: stringr

! Skipping missing files: /gpfs3/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb/utils/modules/R/phasingtools/R/aggr_ser_by_site.R

! Adding files missing in collate: /gpfs3/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb/utils/modules/R/phasingtools/R/aggr_ser_by_pos.R, /gpfs3/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb/utils/modules/R/phasingtools/R/bp_betwen_single_switches.R, /gpfs3/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb/utils/modules/R/phasingtools/R/eval_gt_agreement_by_bin.R, /gpfs3/well/lindgren-ukbb/pro

In [ ]:
frea

In [2]:
args <- list(
    n_samples = 1000,
    pp_cutoff = 0.95,
    seed = 52,
    sites = "/well/lindgren/UKBIOBANK/dpalmer/wes_200k/ukb_wes_qc/data/variants/08_final_qc.keep.variant_list",
    mac_bin = "singleton",
    input_path = "data/prephased/wes_union_calls/phase_conf/ukb_shapeit5_whatshap_chr20.PP.PS.txt.gz",
    out_prefix = "test"

)

In [15]:
# get WES sites
n_samples <- as.numeric(args$n_samples)
pp_cutoff <- as.numeric(args$pp_cutoff)
seed <- as.numeric(args$seed)
sites <- args$sites
mac_bin <- args$mac_bin
input_path <- args$input_path
out_prefix <- args$out_prefix
stopifnot(file.exists(sites))
variants <- fread(sites)

# get phased fil
d <- fread(args$input_path)
stopifnot("MAC" %in% colnames(d))
stopifnot("PP" %in% colnames(d))
keep <- d$locus %in% variants$locus
d <- d[keep,]
d$MAC <- as.numeric(d$MAC)

# bins
bins <- c(0,1,5,10,20,50,100,200,500,1000,2000,5000,10000, Inf)

# keep phased sets with at least one rare variant
bool_keep <- (!is.na(d$PP) & d$MAC < (bins[length(bins)-1]+1))
ps_to_keep <- unique(d$PS_rb[bool_keep])
dt <- d[d$PS_rb %in% ps_to_keep,]
stopifnot(nrow(dt) > 0)

# use same bins as in S5 paper
labels <- unlist(lapply(2:length(bins), function(i){paste0(bins[i-1]+1,"-",bins[i])}))
labels[labels == '1-1'] <- "singleton"
dt$bin <- cut(dt$MAC, breaks = bins, labels = labels)

# ensure that selected mac_bin is available.
stopifnot(mac_bin %in% labels)
labels_to_run <- mac_bin

# NA's are due to well phased genotypes
dt$PP[is.na(dt$PP)] <- 1


In [49]:
n_samples <- 1000
set.seed(seed)
my_samples <- sample(unique(dt$s), n_samples, replace = FALSE)
dt_new <- dt[dt$s %in% my_samples,]

In [50]:
system.time({eval_gt_agreement_by_bin(dt_new, "201-500",pp_cutoff = 0.90, verbose =  TRUE) })

   user  system elapsed 
134.267   6.151 140.847 

ERROR: Error in time$elapsed: $ operator is invalid for atomic vectors


In [ ]:
"2-5" %in% dt_new$bin

In [37]:
eval_gt_agreement_by_bin <- function(dt, labels, pp_cutoff = NULL, verbose = TRUE){
    stopifnot("GT" %in% colnames(dt))
    stopifnot("GT_rb" %in% colnames(dt))
    stopifnot("GT_rb" %in% colnames(dt))
    stopifnot("PP" %in% colnames(dt)) # phasing quality from S5
    stopifnot("rsid" %in% colnames(dt))
    stopifnot("bin" %in% colnames(dt))
    stopifnot(all(labels %in% dt$bin))
    by_bin <- lapply(labels, function(lab){
        if (verbose) write(paste0("Evaluating label ", lab,".."), stderr())
        bool_rsid <-  dt$bin %in% lab
        if (!is.null(pp_cutoff)){
            stopifnot(pp_cutoff <= 1)
            bool_rsid <- bool_rsid & dt$PP >= pp_cutoff
        }
        variants <- unique(dt$rsid[bool_rsid])
        phased_sets <- unique(dt$PS_rb[dt$rsid %in% variants])
        by_ps <- lapply(phased_sets, function(ps){
            bool_sample <- (dt$PS_rb %in% ps)
            samples <- unique(dt$s[bool_sample])
            by_sample <- lapply(samples, function(sid){
                bool_ps_sid <- (dt$s %in% sid) & (dt$PS_rb %in% ps)
                dt_ps_sid <- dt[bool_ps_sid,]
                if (any(variants %in% dt_ps_sid$rsid) & (nrow(dt_ps_sid) == 2)){
                    gt_rb <- dt_ps_sid$GT_rb
                    gt_rb_mirror <- mirror_gt(dt_ps_sid$GT_rb)
                    gt <- dt_ps_sid$GT
                    match <- any(all(gt == gt_rb) | all(gt == gt_rb_mirror))
                    min_mac <- min(dt_ps_sid$MAC)
                    max_mac <- max(dt_ps_sid$MAC)
                    min_pp <- min(dt_ps_sid$PP, na.rm = TRUE)
                    max_pp <- max(dt_ps_sid$PP) # PP is NA for certain varaints
                    max_pp <- ifelse(is.na(max_pp), 1, max_pp)
                    idx_target <- which((dt_ps_sid$PP == min_pp) & (dt_ps_sid$MAC == min_mac))
                    idx_informative <- which((dt_ps_sid$PP == max_pp) & (dt_ps_sid$MAC == max_mac))
                    # if both informative and target SNP has same PP/MAC, randomly select
                    idx_target <- ifelse(length(idx_target)==1, idx_target, 1)
                    idx_informative <- ifelse(length(idx_informative)==1, idx_target, 2)
                    rsid_informative <- dt_ps_sid$rsid[idx_informative]
                    rsid_target <- dt_ps_sid$rsid[idx_target]
                    result <- data.frame(
                        sid = sid,
                        ps = ps,
                        rsid_max_pp = rsid_informative,
                        rsid_min_pp = rsid_target,
                        min_mac = min_mac,
                        max_mac = max_mac,
                        min_pp = min_pp,
                        max_pp = max_pp,
                        match = match,
                        bin = lab
                    )
                    if (nrow(result) > 1){
                        print(head(result))
                        stop("stopped")
                    }
                    return(result)
                }
            })
            by_sample <- do.call(rbind, by_sample)
            return(by_sample)
        })
        by_ps <- do.call(rbind, by_ps)
        return(by_ps)
    })
    final <- do.call(rbind, by_bin)
    return(final)
}

In [14]:
path <- "data/permute/genes/chr7/ukb_eur_wes_200k_pLoF_damaging_missense_chr7_ENSG00000164692.tsv.gz"

In [10]:
phased_calls_500k_command <- "bcftools query -l data/phased/calls/shapeit5/by_maf/ukb_phased_calls_500k_chr21.vcf.gz"

phased_calls_500k_command <- "bcftools query -l data/phased/calls/shapeit5/by_maf/ukb_phased_calls_500k_chr21.vcf.gz"
phased_calls_500k <- fread(cmd = command)
colnames(phased_calls_500k) <- "s"
overlap <- fread("data/unphased/overlap/ukb_calls_wes_samples.txt")

In [12]:
length(intersect(phased_calls_500k$s, overlap$s))

[1] 198857

In [13]:
199282 - 198857

[1] 425

In [15]:
d <- fread(path, tmpdir = "data/tmp/rtmp/")

In [16]:
shuffle_knockouts <- function(d){
    d$KO <- rbinom(n=nrow(d), size=1, prob = d$pTKO)
    d$pKO <- ifelse((d$KO == 1), 1,
             ifelse((d$phased_het == 1 & d$unphased_het > 0), 1 - 1*(1/2)^d$unphased_het,
             ifelse((d$phased_het == 0 & d$unphased_het > 1), 1 - 2*(1/2)^d$unphased_het, 0)))
    return(d$pKO)
}

In [17]:
n = 10

In [19]:
# where is 3717495 (0.5)
# where is 1365568 (0.5)
# where is 1466583 (0.5)
weird_samples <- c(3717495, 1365568, 1466583)

In [20]:
d[d$s %in% c(weird_samples)]

gene_id,s,unphased_het,phased_het,hom_alt_n,pTKO
<chr>,<int>,<int>,<int>,<int>,<dbl>
ENSG00000164692,1365568,1,1,0,0
ENSG00000164692,1466583,2,0,0,0
ENSG00000164692,3717495,1,1,0,0


In [ ]:
#ENSG00000164692	6026401	0	0	0	0.0000e+00
#ENSG00000164692	1326715	0	2	0	5.0000e-01
#ENSG00000164692	1890022	0	2	0	5.0000e-01
#ENSG00000164692	2301892	0	2	0	5.0000e-01
#ENSG00000164692	2314440	0	2	0	5.0000e-01
#ENSG00000164692	5571743	0	2	0	5.0000e-01

# where is 3717495 (0.5)
# where is 1365568 (0.5)
# where is 1466583 (0.5)

In [50]:
reps <- replicate(n, shuffle_knockouts(d))
rownames(reps) <- d$s
reps <- data.table(t(reps))

In [57]:
mat <- reps[,colSums(reps) > 0, with = FALSE]
mat <- mat[rowSums(mat) == 5, ]
ok <- mat[["1326715"]] == 1 & 
      #mat[["1890022"]] == 0 & 
      #mat[["2301892"]] == 0 & 
      mat[["2314440"]] == 0 
mat[ok, ]

#melted_mat <- melt(mat)

1326715,1365568,1466583,1702288,1890022,2301892,2314440,3717495,5571743
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,0.5,0.5,0.5,0,1,0,0.5,1
1,0.5,0.5,0.5,1,1,0,0.5,0
1,0.5,0.5,0.5,1,1,0,0.5,0
1,0.5,0.5,0.5,1,0,0,0.5,1
1,0.5,0.5,0.5,1,0,0,0.5,1
1,0.5,0.5,0.5,1,1,0,0.5,0
1,0.5,0.5,0.5,1,1,0,0.5,0
1,0.5,0.5,0.5,1,0,0,0.5,1
1,0.5,0.5,0.5,1,1,0,0.5,0


In [52]:
#mat

1326715,1365568,1466583,1702288,1890022,2301892,2314440,3717495,5571743
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
0,0.5,0.5,0.5,0,1,1,0.5,1
0,0.5,0.5,0.5,1,1,0,0.5,1
0,0.5,0.5,0.5,1,1,0,0.5,1
1,0.5,0.5,0.5,1,0,1,0.5,0
0,0.5,0.5,0.5,0,1,1,0.5,1
0,0.5,0.5,0.5,1,0,1,0.5,1
0,0.5,0.5,0.5,1,1,0,0.5,1
1,0.5,0.5,0.5,0,1,0,0.5,1
0,0.5,0.5,0.5,0,1,1,0.5,1


In [9]:
p99 <- "data/permute/permutations/chr7/ENSG00000164692/ukb_eur_wes_200k_pLoF_damaging_missense_permuted_chr7_ENSG00000164692_99.vcf.gz"
p98 <- "data/permute/permutations/chr7/ENSG00000164692/ukb_eur_wes_200k_pLoF_damaging_missense_permuted_chr7_ENSG00000164692_98.vcf.gz"

In [10]:
d99 <- fread(p99, tmpdir = "data/tmp/rtmp/", skip = '##', nrows = 10)
d98 <- fread(p98, tmpdir = "data/tmp/rtmp/", skip = '##', nrows = 10)

In [13]:
d99[1,]
d98[1,]

#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,1000028,⋯,6026237,6026268,6026270,6026295,6026311,6026322,6026335,6026383,6026397,6026401
<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
chr7,1,ENSG00000164692,X,Y,.,.,.,DS,0,⋯,0,0,0,0,0,0,0,0,0,0


#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,1000028,⋯,6026237,6026268,6026270,6026295,6026311,6026322,6026335,6026383,6026397,6026401
<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
chr7,1,ENSG00000164692,X,Y,.,.,.,DS,0,⋯,0,0,0,0,0,0,0,0,0,0


In [89]:
head(d, 2)

#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,1000028,⋯,6026237,6026268,6026270,6026295,6026311,6026322,6026335,6026383,6026397,6026401
<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
chr7,1,ENSG00000164692,X,Y,.,.,.,DS,0,⋯,0,0,0,0,0,0,0,0,0,0
chr7,2,ENSG00000164692,X,Y,.,.,.,DS,0,⋯,0,0,0,0,0,0,0,0,0,0


In [83]:
table(d$p.value)


0.653154 0.707262 0.723476 0.725199  0.80211 0.894845 0.903404 0.913869 
   62506    62443    62469    62624    62550    62412    62833    62593 
0.922451 0.968206 0.975673 
   62163    62491    62413 

In [77]:
fread()

V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,⋯,V161275,V161276,V161277,V161278,V161279,V161280,V161281,V161282,V161283,V161284
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,1000028,⋯,5584149,5584165,5584201,5584225,5584299,5584312,5584377,5584402,5584416,55


In [52]:
args <- list(
    phenotype_bin = "data/phenotypes/spiros_brava_phenotypes_binary_500k.tsv",
    directory = "data/prs/scores/"
)



In [64]:
print(args)
stopifnot(file.exists(args$phenotype_bin))
stopifnot(dir.exists(args$directory))


# Load polygenic risk scores
files <- list.files(args$directory, pattern = ".txt.gz", full.names = TRUE)
files <- files[!grepl("chrom",files)]
print(files)

# combine files
lst <- lapply(files, fread)
mrg <- Reduce(merge, lst)
mrg$eid <- mrg$sid

# read binary phenotypes
d_bin <- fread(args$phenotype_bin)
d_bin <- d_bin[d_bin$eid %in% mrg$eid,]
cols_bin <- colnames(d_bin)

# subset cols to binary
cols <- gsub("_pgs","",colnames(mrg))
cols <- cols[cols %in% cols_bin]
d_bin <- d_bin[,colnames(d_bin) %in% cols, with = FALSE]
dt <- merge(mrg, d_bin, by = 'eid')

# calculate AUC
cols <- cols[!cols %in% "eid"]


$phenotype_bin
[1] "data/phenotypes/spiros_brava_phenotypes_binary_500k.tsv"

$directory
[1] "data/prs/scores/"

  [1] "data/prs/scores//AD_combined_pgs.txt.gz"                                                            
  [2] "data/prs/scores//AD_combined_primary_care_pgs.txt.gz"                                               
  [3] "data/prs/scores//AFR_int_pgs.txt.gz"                                                                
  [4] "data/prs/scores//AMRA_int_pgs.txt.gz"                                                               
  [5] "data/prs/scores//AUT_combined_pgs.txt.gz"                                                           
  [6] "data/prs/scores//AUT_combined_primary_care_pgs.txt.gz"                                              
  [7] "data/prs/scores//Alanine_aminotransferase_residual_int_pgs.txt.gz"                                  
  [8] "data/prs/scores//Albumin_residual_int_pgs.txt.gz"                                                   
  [9] "data/prs/scores/

In [57]:
x1 <- "spiro_diabetes"
x2 <- "DM_T2D"

In [61]:
col <- x2
write(paste("calculating auc for", col), stderr())
col_pgs <- paste0(col,'_pgs')
boot <- dt[,colnames(dt) %in% c('eid',col,col_pgs), with = FALSE]
boot <- boot[!is.na(boot[[col]]) & !is.na(boot[[col_pgs]]),]



In [63]:
boot

eid,DM_T2D_pgs,DM_T2D
<int>,<int>,<lgl>
1000028,0,FALSE
1000034,0,FALSE
1000087,0,FALSE
1000118,0,FALSE
1000120,0,FALSE
1000171,0,FALSE
1000196,0,FALSE
1000254,0,FALSE
1000278,0,FALSE


In [ ]:
auc <- AUCBoot(
    pred = boot[[col_pgs]],
    target = as.numeric(boot[[col]]),
    nboot = 1000,
    seed = 1995
)

cases <- sum(boot[[col]]==1)
controls <- sum(boot[[col]]==0)
boot <- data.table(t(auc))
colnames(boot) <- paste0("auc_",tolower(colnames(boot)))
colnames(boot) <- gsub("\\%","_pct", colnames(boot))
colnames(boot) <- gsub("\\.","_", colnames(boot))
boot$pred_cases <- cases
boot$pred_controls <- controls
boot$pred_n <- cases + controls
boot$phenotype <- col

